In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from transformers import (
    RobertaTokenizer, 
    RobertaForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Load your local TSV
# (Make sure 'dontpatronizeme_v1.2.tsv' is in the exact same folder as this script)
full_df = pd.read_csv('dontpatronizeme_pcl.tsv', sep='\t', 
                     names=['par_id', 'art_id', 'keyword', 'country', 'text', 'label'])

In [ ]:
# Drop rows missing text to prevent the ArrowTypeError
full_df = full_df.dropna(subset=['text'])

# Convert labels: 0, 1 -> 0 (No PCL) | 2, 3, 4 -> 1 (Yes PCL)
full_df['label'] = full_df['label'].apply(lambda x: 1 if x >= 2 else 0)

# Force the text column to string to prevent float errors
full_df['text'] = full_df['text'].astype(str)

In [ ]:
# Split: 80% Train, 20% Validation (Dev)
# stratify ensures the 20% has the same proportion of PCL cases as the 80%
train_raw, dev_raw = train_test_split(
    full_df, test_size=0.2, random_state=42, stratify=full_df['label']
)

# Downsample the Training set only (Ratio: 1 Positive to 2 Negative)
pos_train = train_raw[train_raw.label == 1]
neg_train = train_raw[train_raw.label == 0].sample(n=len(pos_train)*2, random_state=42)
train_df = pd.concat([pos_train, neg_train]).sample(frac=1, random_state=42) # Shuffle

print(f"Training set size: {len(train_df)} (Pos: {len(pos_train)}, Neg: {len(neg_train)})")
print(f"Validation set size: {len(dev_raw)}")

In [ ]:
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    # return tokenizer(batch["text"], truncation=True, padding=False)
    return tokenizer(batch["text"], truncation=True, max_length=128, padding=False)

# Convert Pandas DataFrames to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(tokenize_fn, batched=True)
dev_ds = Dataset.from_pandas(dev_raw, preserve_index=False).map(tokenize_fn, batched=True)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    # Calculate specifically for the PCL class (pos_label=1)
    f1 = f1_score(labels, preds, pos_label=1, average='binary')
    prec = precision_score(labels, preds, pos_label=1, average='binary')
    rec = recall_score(labels, preds, pos_label=1, average='binary')
    acc = accuracy_score(labels, preds)
    
    return {"f1_pcl": f1, "precision": prec, "recall": rec, "accuracy": acc}

In [ ]:
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        weights = torch.tensor([1.0, 4.0], dtype=torch.float32, device=logits.device)
        
        loss = F.cross_entropy(
            logits.view(-1, model.config.num_labels), 
            labels.view(-1),
            weight=weights
        )
        
        return (loss, outputs) if return_outputs else loss

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # gradient accumulation to stop running out of vram
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer, 
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
print("\nStarting training...")
trainer.train()

print("\nEvaluating baseline on Validation Set...")
baseline_results = trainer.evaluate()

print("\n" + "="*30)
print("FINAL VALIDATION RESULTS")
print("="*30)
for key, value in baseline_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")